<a href="https://colab.research.google.com/github/masteryongsa82/Portfolio/blob/main/Cipher_Version2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import hashlib, random, os, base64

def sha256(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

def make_permutation(seed: bytes) -> list:
    """시드로부터 결정론적 순열 생성"""
    rng = random.Random(int.from_bytes(seed[:8], 'big'))
    perm = list(range(26))
    rng.shuffle(perm)
    return perm

def invert(perm: list) -> list:
    """순열의 역치환"""
    inv = [0] * 26
    for i, v in enumerate(perm):
        inv[v] = i
    return inv

def rol32(v, n): return ((v << n) | (v >> (32 - n))) & 0xFFFFFFFF

class AsymmetricEnigmaKeyPair:

    N_ROTORS = 5

    def __init__(self, master_seed: bytes = None):
        if master_seed is None:
            master_seed = os.urandom(32)
        self.master_seed = master_seed

        pub_seed = sha256(b"public_rotors:" + master_seed)
        self.public_rotors = []
        for i in range(self.N_ROTORS):
            rotor_seed = sha256(pub_seed + i.to_bytes(4, 'big'))
            self.public_rotors.append(make_permutation(rotor_seed))

        priv_seed = sha256(b"private_rotors:" + master_seed)
        self.private_rotors = []
        for i in range(self.N_ROTORS):
            rotor_seed = sha256(priv_seed + i.to_bytes(4, 'big'))
            self.private_rotors.append(make_permutation(rotor_seed))

        self.private_rotors = [invert(r) for r in self.public_rotors]

        pb_seed = sha256(b"plugboard:" + master_seed)
        pb_perm = make_permutation(pb_seed)
        self.plugboard = list(range(26))
        for i in range(0, 20, 2):
            a, b = pb_perm[i], pb_perm[i+1]
            self.plugboard[a] = b
            self.plugboard[b] = a

        sched_seed = sha256(b"schedule:" + master_seed)
        self.step_schedule = self._build_schedule(sched_seed, 100000)

        self.public_key = {
            'rotors':    self.public_rotors,
            'plugboard': self.plugboard,
            'schedule':  self.step_schedule,
        }

        self.private_key = {
            'rotors': self.private_rotors,
        }

    def _build_schedule(self, seed: bytes, length: int) -> list:
        schedule = []
        h = seed
        while len(schedule) < length:
            h = sha256(h)
            for b in h:
                schedule.append((b % 25) + 2)
        return schedule[:length]

    def show(self):
        print("── 키 정보 ──")
        print(f"  마스터 시드: {self.master_seed.hex()[:20]}...")
        print(f"  공개 회전자 0: {self.public_rotors[0][:10]}...")
        print(f"  개인 회전자 0: {self.private_rotors[0][:10]}...")
        print(f"  플러그보드  : {self.plugboard[:10]}...")

        r = self.public_rotors[0]
        ir = self.private_rotors[0]
        sample = [0, 5, 10, 15]
        print(f"  [검증] 공개[i]=v 이면 개인[v]=i:")
        for i in sample:
            v = r[i]
            print(f"    공개[{i:2d}]={v:2d}, 개인[{v:2d}]={ir[v]:2d}  {'OK' if ir[v] == i else 'ERROR'}")


class AsymmetricEnigma:

    def __init__(self, key_pair: AsymmetricEnigmaKeyPair):
        self.kp = key_pair
        self._enc_positions = [0] * key_pair.N_ROTORS
        self._dec_positions = [0] * key_pair.N_ROTORS
        self._enc_sched_idx = 0
        self._dec_sched_idx = 0

    def _step(self, positions, sched_idx):
        """키 의존 불규칙 회전"""
        positions[0] = (positions[0] + 1) % 26
        for i in range(1, len(positions)):
            interval = self.kp.step_schedule[sched_idx % len(self.kp.step_schedule)]
            sched_idx += 1
            if positions[0] % interval == 0:
                positions[i] = (positions[i] + 1) % 26
        return sched_idx

    def reset(self):
        self._enc_positions = [0] * self.kp.N_ROTORS
        self._dec_positions = [0] * self.kp.N_ROTORS
        self._enc_sched_idx = 0
        self._dec_sched_idx = 0

    def encrypt_char(self, c: int) -> int:
        """공개키로 암호화: 플러그보드 → 공개 회전자들 순방향"""
        self._enc_sched_idx = self._step(self._enc_positions, self._enc_sched_idx)
        c = self.kp.plugboard[c]
        for i, rotor in enumerate(self.kp.public_rotors):
            c = rotor[(c + self._enc_positions[i]) % 26]
        return c

    def decrypt_char(self, c: int) -> int:
        """개인키로 복호화: 개인 회전자들 역방향 → 플러그보드"""
        self._dec_sched_idx = self._step(self._dec_positions, self._dec_sched_idx)
        for i in range(len(self.kp.private_rotors) - 1, -1, -1):
            rotor = self.kp.private_rotors[i]
            c = (rotor[c] - self._dec_positions[i] + 26) % 26
        c = self.kp.plugboard[c]
        return c

    def encrypt(self, text: str) -> str:
        result = []
        for ch in text.upper():
            if ch.isalpha():
                result.append(chr(self.encrypt_char(ord(ch) - 65) + 65))
            else:
                result.append(ch)
        return ''.join(result)

    def decrypt(self, text: str) -> str:
        result = []
        for ch in text.upper():
            if ch.isalpha():
                result.append(chr(self.decrypt_char(ord(ch) - 65) + 65))
            else:
                result.append(ch)
        return ''.join(result)


if __name__ == '__main__':

    # 메인 실행
    print("비대칭 에니그마 시뮬레이션")

    kp = None
    enigma = None

    while True:
        print("\n--- 메뉴 ---")
        print("1. 새 키 쌍 생성")
        print("2. 암호화 (공개키 사용)")
        print("3. 복호화 (개인키 사용)")
        print("4. 키 정보 표시")
        print("5. 종료")

        choice = input("선택: ")

        if choice == '1':
            seed_input = input("마스터 시드를 입력하세요 (생략하면 랜덤): ")
            if seed_input:
                kp = AsymmetricEnigmaKeyPair(seed_input.encode())
            else:
                kp = AsymmetricEnigmaKeyPair()
            enigma = AsymmetricEnigma(kp)
            print("새 키 쌍이 생성되었습니다.")
        elif choice == '2':
            if enigma is None:
                print("먼저 키 쌍을 생성해주세요.")
                continue
            message = input("암호화할 메시지를 입력하세요: ")
            enigma.reset() # 매번 암호화/복호화 전에 초기화
            encrypted_message = enigma.encrypt(message)
            print(f"암호화된 메시지: {encrypted_message}")
        elif choice == '3':
            if enigma is None:
                print("먼저 키 쌍을 생성해주세요.")
                continue
            encrypted_message = input("복호화할 메시지를 입력하세요: ")
            enigma.reset() # 매번 암호화/복호화 전에 초기화
            decrypted_message = enigma.decrypt(encrypted_message)
            print(f"복호화된 메시지: {decrypted_message}")
        elif choice == '4':
            if kp is None:
                print("먼저 키 쌍을 생성해주세요.")
                continue
            kp.show()
        elif choice == '5':
            print("종료합니다.")
            break
        else:
            print("잘못된 선택입니다. 다시 시도해주세요.")

비대칭 에니그마 시뮬레이션

--- 메뉴 ---
1. 새 키 쌍 생성
2. 암호화 (공개키 사용)
3. 복호화 (개인키 사용)
4. 키 정보 표시
5. 종료
선택: 1
마스터 시드를 입력하세요 (생략하면 랜덤): 010
새 키 쌍이 생성되었습니다.

--- 메뉴 ---
1. 새 키 쌍 생성
2. 암호화 (공개키 사용)
3. 복호화 (개인키 사용)
4. 키 정보 표시
5. 종료
선택: 2
암호화할 메시지를 입력하세요: Hello
암호화된 메시지: XBSBI

--- 메뉴 ---
1. 새 키 쌍 생성
2. 암호화 (공개키 사용)
3. 복호화 (개인키 사용)
4. 키 정보 표시
5. 종료
선택: 3
복호화할 메시지를 입력하세요: XBSBI
복호화된 메시지: HELLO

--- 메뉴 ---
1. 새 키 쌍 생성
2. 암호화 (공개키 사용)
3. 복호화 (개인키 사용)
4. 키 정보 표시
5. 종료
선택: 1
마스터 시드를 입력하세요 (생략하면 랜덤): 101
새 키 쌍이 생성되었습니다.

--- 메뉴 ---
1. 새 키 쌍 생성
2. 암호화 (공개키 사용)
3. 복호화 (개인키 사용)
4. 키 정보 표시
5. 종료
선택: 3
복호화할 메시지를 입력하세요: XBSBI
복호화된 메시지: YISNP

--- 메뉴 ---
1. 새 키 쌍 생성
2. 암호화 (공개키 사용)
3. 복호화 (개인키 사용)
4. 키 정보 표시
5. 종료
선택: 1
마스터 시드를 입력하세요 (생략하면 랜덤): 010
새 키 쌍이 생성되었습니다.

--- 메뉴 ---
1. 새 키 쌍 생성
2. 암호화 (공개키 사용)
3. 복호화 (개인키 사용)
4. 키 정보 표시
5. 종료
선택: XBSBI
잘못된 선택입니다. 다시 시도해주세요.

--- 메뉴 ---
1. 새 키 쌍 생성
2. 암호화 (공개키 사용)
3. 복호화 (개인키 사용)
4. 키 정보 표시
5. 종료
선택: 2
암호화할 메시지를 입력하세요: 
암호화된 메시지: 

--- 메뉴 ---
1. 새 키 쌍 생성
2. 암호화 (공개키 사용)
3. 복호화